In [48]:
import os
import camelot
import pandas as pd
from tqdm import tqdm
from IPython.display import display
import warnings
import re
import numpy as np
import pdfplumber

warnings.filterwarnings("ignore")

folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


def extract_numbers(name):

    nums = re.findall(r'\d+', name)
    return tuple(map(int, nums))

replacements = {
    'Ⅰ': 'I',
    'Ⅱ': 'II',
    'Ⅲ': 'III',
    'Ⅳ': 'IV',
    'Ⅴ': 'V',
    'Ⅵ': 'VI',
    'Ⅶ': 'VII',
    'Ⅷ': 'VIII',
    'Ⅸ': 'IX',
    'Ⅹ': 'X'
}

def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):

    os.makedirs(out_dir, exist_ok=True)

    tables_lattice = camelot.read_pdf(
        pdf_path,
        pages=pages,
        flavor="lattice",
        line_scale=80
    )

    lattice_pages = set()
    text_pages = set()

    with pdfplumber.open(pdf_path) as pdf:

        for i, t in enumerate(tables_lattice):

            page_num = int(t.page)          
            page_idx = page_num - 1        
            lattice_pages.add(page_num)

            t.to_csv(
                os.path.join(out_dir, f"lattice_p{page_num}_{i}.csv"),
                index=False
            )

            page = pdf.pages[page_idx]

            x1, y1, x2, y2 = t._bbox
            page_w = page.width
            page_h = page.height

            table_top = page_h - y2

            left   = max(0, min(x1, page_w))
            right  = max(0, min(x2, page_w))
            top    = max(0, min(table_top - title_height, page_h))
            bottom = max(0, min(table_top, page_h))

            if right > left and bottom > top:
                title_box = (left, top, right, bottom)
                title_text = page.crop(title_box, strict=False).extract_text(
                    x_tolerance=2,
                    y_tolerance=2
                ) or ""

                if title_text.strip():
                    text_pages.add(page_num)
            else:
                title_text = ""

            with open(
                os.path.join(out_dir, f"text_p{page_num}_{i}.txt"),
                "w",
                encoding="utf-8"
            ) as f:
                f.write(title_text.strip())

    return None       



def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None



def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'Qty' in df.columns:
        agg_map = {
            "Qty": "sum",
        }
    elif 'Код материалов' in df.columns:
        agg_map = {
            "Кол-во": "sum",
        }
    elif '件数' in df.columns:
        agg_map = {
            '件数': 'sum'
        }
    elif 'Units' in df.columns:
        agg_map = {
            'Units': 'sum'
        }

    if "Item and Spec" in df.columns:

        agg_map["Item and Spec"] = join_ordered
    if "Remark" in df.columns:
        agg_map["Remark"] = join_ordered

    if 'Наименование и спецификация' in df.columns:
        agg_map['Наименование и спецификация'] = join_ordered

    if 'Примечание' in df.columns:
        agg_map['Примечание'] = join_ordered

    if 'Версия' in df.columns:
        agg_map['Версия'] = join_ordered

    if 'Rev.' in df.columns:
        agg_map['Rev.'] = join_ordered

    if "名称及规格" in df.columns:
        agg_map["名称及规格"] = join_ordered

    if "备注" in df.columns:
        agg_map["备注"] = join_ordered

    if "Section" in df.columns:
        agg_map["Section"] = join_ordered


    if 'Material No.' in df.columns:
        df_agg = df.groupby("Material No.", as_index=False).agg(agg_map)
    elif 'Код материалов' in df.columns:
        df_agg = df.groupby("Код материалов", as_index=False).agg(agg_map)
    elif '物料编码' in df.columns:
        df_agg = df.groupby("物料编码", as_index=False).agg(agg_map)


    return df_agg

b = 18
e = b+1

for i, pth in enumerate(tqdm(pdf_files[b:e])):
    
    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]
    print(f'Обработка файла: {filename}')

    # extract_tables_camelot(pth, filename, pages="all", title_height=800)

    dfs = []

    for file in sorted(os.listdir(filename), key=extract_numbers):

        if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
            continue

        path = os.path.join(filename, file)

        if os.path.getsize(path) == 0:  
            continue
        df_i = pd.read_csv(path, converters={1: str})

        required_cols = ['NO.', 'Код материалов', 'No.', 'Код мат-ла', 'Serial No.', 'Part No.', '物料编码']

        if (
            df_i.empty
            or df_i.shape[1] < 4
            or not any(col in df_i.columns for col in required_cols)
            ):
            continue

        # если китаец насрал пробел в начале таблицы
        if 'NO.' in df_i.iloc[0].values or 'Material No.' in df_i.iloc[0].values:
            df_i.columns = df_i.iloc[0]  
            df_i = df_i[1:].reset_index(drop=True) 

        if 'NO.' in df_i.columns:
            df_i = df_i.drop("NO.", axis=1)
        if 'No.' in df_i.columns:
            df_i = df_i.drop("No.", axis=1)
        elif '№ п/п' in df_i.columns:
            df_i = df_i.drop("№ п/п", axis=1)
        elif '№ \nп/п' in df_i.columns:
            df_i = df_i.drop('№ \nп/п', axis=1)
        elif 'Serial No.' in df_i.columns:
            df_i = df_i.drop('Serial No.', axis=1)
        elif '序号' in df_i.columns:
            df_i = df_i.drop('序号', axis=1)

        idx = file.replace("lattice_", "").rsplit(".", 1)[0]
        section_val = os.path.join(filename, f"text_{idx}.txt")
        
        with open(section_val, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        skip_titles = [
            'Руководство по деталъная структура',
            'Альбом чертежей деталей и узлов коленчатого подъемника для высотных работ',
            'Альбом чертежей деталей и компонентов',
            'Parts Manual'
        ]

        if lines and lines[0] in skip_titles:
            lines = lines[1:]
                
        if lines and lines[0] == 'Альбом чертежей деталей и узлов коленчатого':
            lines = lines[2:]

        lines_joined = " ".join(lines)
            
        df_i["Section"] = lines_joined
        

        df_i = df_i[df_i.iloc[:, 0].replace('', pd.NA).notna()]
        display(df_i)
        df_i = df_i[df_i.iloc[:, 0] != '/']
        dfs.append(df_i)

    df_all = pd.concat(dfs, ignore_index=True)
    display(df_all)
    df_all.to_excel('dfs.xlsx', index=False)

    for old_col in ['Наименование и \nспецификация', 'Наименование и \r\nспецификация']:
        if old_col in df_all.columns:
            if 'Наименование и спецификация' in df_all.columns:
                df_all['Наименование и спецификация'] = df_all.loc[:, old_col].combine_first(df_all['Наименование и спецификация'])
            else:
                df_all['Наименование и спецификация'] = df_all[old_col]
            df_all = df_all.drop(columns=old_col)
    if 'Наименование и спецификация' in df_all.columns:
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col) 
                
    if 'Name and Spec.' in df_all.columns:
        if 'Item and Spec' in df_all.columns:
            df_all['Item and Spec'] = df_all.loc[:, 'Name and Spec.'].combine_first(df_all['Item and Spec'])
        else:
            df_all['Item and Spec'] = df_all['Name and Spec.']
        df_all = df_all.drop(columns='Name and Spec.')
        col = df_all.pop("Item and Spec")  
        df_all.insert(1, "Item and Spec", col)  
        
    if 'Spec and Item' in df_all.columns:
        if 'Item and Spec' in df_all.columns:
            df_all['Item and Spec'] = df_all.loc[:, 'Spec and Item'].combine_first(df_all['Item and Spec'])
        else:
            df_all['Item and Spec'] = df_all['Spec and Item']
        df_all = df_all.drop(columns='Spec and Item')
        col = df_all.pop("Item and Spec")  
        df_all.insert(1, "Item and Spec", col)  

    if 'Версия  Кол-' in df_all.columns:
        df_all['Версия'] = df_all.loc[:, 'Версия  Кол-'].combine_first(df_all['Версия'])
        df_all = df_all.drop(columns='Версия  Кол-')

    if 'во' in df_all.columns:
        df_all['Кол-\nво'] = df_all.loc[:, 'во'].combine_first(df_all['Кол-\nво'])
        df_all = df_all.drop(columns='во')

    if 'Nombre y especificación' in df_all.columns:
        df_all['Наименование и спецификация'] = df_all.loc[:, 'Nombre y especificación'].combine_first(df_all['Наименование и спецификация'])
        df_all = df_all.drop(columns='Nombre y especificación')
        col = df_all.pop("Наименование и спецификация")  
        df_all.insert(1, "Наименование и спецификация", col)  
    
    if 'Примечани\nе' in df_all.columns:
        df_all['Примечание'] = df_all.loc[:, 'Примечани\nе'].combine_first(df_all['Примечание'])
        df_all = df_all.drop(columns='Примечани\nе')

    if 'Part No.' in df_all.columns:
        if 'Material No.' in df_all.columns:
            df_all['Material No.'] = df_all.loc[:, 'Part No.'].combine_first(df_all['Material No.'])
        else:
            df_all['Material No.'] = df_all['Part No.']
        df_all = df_all.drop(columns='Part No.')
        col = df_all.pop("Material No.")  
        df_all.insert(0, "Material No.", col)  

    if 'Código de material' in df_all.columns:
        df_all['Код материалов'] = df_all.loc[:, 'Código de material'].combine_first(df_all['Код материалов'])
        df_all = df_all.drop(columns='Código de material')
        col = df_all.pop("Код материалов")  
        df_all.insert(0, "Код материалов", col)  

    if 'Кол-\nво' in df_all.columns:
        if 'Кол-во' in df_all.columns:
            df_all['Кол-во'] = df_all.loc[:, 'Кол-\nво'].combine_first(df_all['Кол-во'])
        else:
            df_all['Кол-во'] = df_all['Кол-\nво']
        df_all = df_all.drop(columns='Кол-\nво')
        col = df_all.pop('Кол-во')  
        df_all.insert(2, 'Кол-во', col)  

    if 'Qty.' in df_all.columns:
        if 'Qty' in df_all.columns:
            df_all['Qty'] = df_all.loc[:, 'Qty.'].combine_first(df_all['Qty'])
        else:
            df_all['Qty'] = df_all['Qty.']
        df_all = df_all.drop(columns='Qty.')
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col)  

    if 'Кол-\r\nво' in df_all.columns:
        if 'Кол-во' in df_all.columns:
            df_all['Кол-во'] = df_all.loc[:, 'Кол-\r\nво'].combine_first(df_all['Кол-во'])
        else:
            df_all['Кол-во'] = df_all['Кол-\r\nво']
        df_all = df_all.drop(columns='Кол-\r\nво')
        col = df_all.pop('Кол-во')  
        df_all.insert(2, 'Кол-во', col)  

    if 'Cantidad' in df_all.columns:
        df_all['Кол-во'] = df_all.loc[:, 'Cantidad'].combine_first(df_all['Кол-во'])
        df_all = df_all.drop(columns='Cantidad')
        col = df_all.pop('Кол-во')  
        df_all.insert(2, 'Кол-во', col)  

    if 'Unit' in df_all.columns:
        df_all['Qty'] = df_all.loc[:, 'Unit'].combine_first(df_all['Qty'])
        df_all = df_all.drop(columns='Unit')
        col = df_all.pop('Qty')  
        df_all.insert(2, 'Qty', col)  

    if 'Note' in df_all.columns and 'Remark' in df_all.columns:
        df_all['Remark'] = df_all.loc[:, 'Note'].combine_first(df_all['Remark'])
        df_all = df_all.drop(columns='Note')
        col = df_all.pop('Remark')  
        df_all.insert(2, 'Remark', col)  
    elif 'Note' in df_all.columns and 'Remark' not in df_all.columns:
        df_all['Remark'] = df_all['Note']
        df_all = df_all.drop(columns='Note')
        col = df_all.pop('Remark')  
        df_all.insert(3, 'Remark', col)  

    if 'Observaciones' in df_all.columns and 'Примечание' in df_all.columns:
        df_all['Примечание'] = df_all.loc[:, 'Observaciones'].combine_first(df_all['Примечание'])
        df_all = df_all.drop(columns='Observaciones')
        col = df_all.pop('Примечание')  
        df_all.insert(3, 'Примечание', col) 

    if 'Кол-в\nо \nштук' in df_all.columns and 'Кол-во \nштук' in df_all.columns:
        df_all['Кол-во \nштук'] = df_all.loc[:, 'Кол-в\nо \nштук'].combine_first(df_all['Кол-во \nштук'])
        df_all = df_all.drop(columns='Кол-в\nо \nштук')

    if 'Кол-\nво \nшту\nк' in df_all.columns and 'Кол-во \nштук' in df_all.columns:
        df_all['Кол-во \nштук'] = df_all.loc[:, 'Кол-\nво \nшту\nк'].combine_first(df_all['Кол-во \nштук'])
        df_all = df_all.drop(columns='Кол-\nво \nшту\nк')

    if 'Кол-во \nштук' in df_all.columns and 'Кол-во' not in df_all.columns:
        df_all['Кол-во'] = df_all['Кол-во \nштук']
        df_all = df_all.drop(columns='Кол-во \nштук')
        col = df_all.pop('Кол-во')  
        df_all.insert(2, 'Кол-во', col)  
    
    if 'Код мат-ла' in df_all.columns and 'Код материалов' not in df_all.columns:
        df_all['Код материалов'] = df_all['Код мат-ла']
        df_all = df_all.drop(columns='Код мат-ла')
        col = df_all.pop('Код материалов')  
        df_all.insert(0, 'Код материалов', col)  

    if 'Кол-во' in df_all.columns:
        df_all['Кол-во'] = df_all['Кол-во'].replace('/', 0)
        df_all['Кол-во'] = df_all['Кол-во'].fillna(0)
        df_all['Кол-во'] = df_all['Кол-во'].astype(int)

    if 'Qty' in df_all.columns:
        df_all['Qty'] = df_all['Qty'].replace('/', 0)
        df_all['Qty'] = df_all['Qty'].fillna(0)
        df_all['Qty'] = df_all['Qty'].astype(int)

    if '件数' in df_all.columns:
        df_all['件数'] = df_all['件数'].replace('/', 0)
        df_all['件数'] = df_all['件数'].replace('、', 0)
        df_all['件数'] = df_all['件数'].fillna(0)
        df_all['件数'] = df_all['件数'].astype(int)

    df_all.to_excel('df_all.xlsx', index=False)
    df_all = pd.read_excel('df_all.xlsx')
    display(df_all.head())

    test_1 = pd.read_excel('dfs.xlsx')
    assert test_1.shape[0] == df_all.shape[0]

    if 'Material No.' in df_all.columns:
        df_all['Material No.'] = df_all['Material No.'].str.split('\n').str[0]
        df_all = df_all[~df_all['Material No.'].astype(str).str.startswith('NO-NUMBER')]

    cols = df_all.select_dtypes(include='object').columns
    df_all[cols] = (df_all[cols].replace(r'\s*\n\s*', ' ', regex=True))
    df_all[cols] = df_all[cols].replace(replacements, regex=True)

    if 'Item and Spec' in df_all.columns:
        df_all['Item and Spec'] = df_all['Item and Spec'].replace('/', np.nan)
    
    if '物料编码' in df_all.columns:
        mask = df_all['物料编码'].str.contains(r'^\d+\s+\D', na=False)
        extracted = df_all.loc[mask, '物料编码'].str.extract(r'^(\d+)\s+(.*)$')
        df_all.loc[mask, '物料编码'] = extracted[0]
        df_all.loc[mask, '名称及规格'] = extracted[1]

    df_merged = merge_parts(df_all) 
    assert (df_all.shape[0] - df_all.iloc[:, 0].duplicated().sum() - df_merged.shape[0]) == 0

    lst = ['ZS1623RT', 'ZA10RJE', 'ZA14J', 'ZA14JE-Li', 'ZA14NJE', 'ZA16J', 'ZA16JERT', 'ZA18J', 'ZA20J', 'ZA20JE', 'ZA20JERT', 'ZA22J-V' ,'ZA22J', 'ZA24J', 'ZA32J(H-образ.)', 'ZA32J(X-образ.)', 'ZT42J-V', 'ZA42J', 'lol']
    df_merged['Models'] = lst[b]

    df_merged.to_excel(f'{filename}.xlsx', index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: ZS 0407 - 1414 HDS


,Material No.,Item and Spec,Units,Remark,Section
0,00775700000001040,Lower sliding block I,1,NaN,ZS1414 Series Pin Installation
1,00775600300001140,Gasket,2,NaN,ZS1414 Series Pin Installation
2,1054400310,Composite bush,2,NaN,ZS1414 Series Pin Installation
3,1040200504,Nut M10-8,2,GB/T889.1-2000,ZS1414 Series Pin Installation
4,1040000025,Bolt M10×80-8.8,2,GB/T5783-2000,ZS1414 Series Pin Installation
5,00775700000001020,Lower sliding axle,1,NaN,ZS1414 Series Pin Installation
6,00775700000001050,Lower sliding block II,1,NaN,ZS1414 Series Pin Installation
7,00775600000001070,Upper stationary axle,4,NaN,ZS1414 Series Pin Installation
8,00775700000001060,Upper sliding block,2,NaN,ZS1414 Series Pin Installation
9,1040300526,Washer 12,8,GB/T97.1-2002,ZS1414 Series Pin Installation


,Material No.,Item and Spec,Units,Remark,Section
0,00775700000001040,Lower sliding block I,1,NaN,ZS1212 Series Pin Installation
1,1040000025,Bolt M10×80-8.8,2,GB/T5783-2000,ZS1212 Series Pin Installation
2,1040200504,Nut M10-8,2,GB/T889.1-2000,ZS1212 Series Pin Installation
3,00775600000001040,Lower sliding axle,1,NaN,ZS1212 Series Pin Installation
4,00775700000001050,Lower sliding block II,1,NaN,ZS1212 Series Pin Installation
5,00775600000001070,Upper stationary axle,4,NaN,ZS1212 Series Pin Installation
6,1040000807,Bolt M12×110-8.8,4,GB/T5782-2000,ZS1212 Series Pin Installation
7,1040300526,Washer 12,8,GB/T97.1-2002,ZS1212 Series Pin Installation
8,00775700000001060,Upper sliding block,2,NaN,ZS1212 Series Pin Installation
9,1040200507,Nut M12-8,4,GB/T889.1-2000,ZS1212 Series Pin Installation


,Material No.,Item and Spec,Units,Remark,Section
0,00775700000001040,Lower sliding block I,1,NaN,ZS1012 Series Pin Installation
1,1040000025,Bolt M10×80-8.8,2,GB/T5783-2000,ZS1012 Series Pin Installation
2,1040200504,Nut M10-8,2,GB/T889.1-2000,ZS1012 Series Pin Installation
3,00775600000001040,Lower sliding axle,1,NaN,ZS1012 Series Pin Installation
4,00775700000001050,Lower sliding block II,1,NaN,ZS1012 Series Pin Installation
5,00775600000001070,Upper stationary axle,4,NaN,ZS1012 Series Pin Installation
6,1040000807,Bolt M12×110-8.8,4,GB/T5782-2000,ZS1012 Series Pin Installation
7,1040300526,Washer 12,8,GB/T97.1-2002,ZS1012 Series Pin Installation
8,00775700000001060,Upper sliding block,2,NaN,ZS1012 Series Pin Installation
9,1040200507,Nut M12-8,4,GB/T889.1-2000,ZS1012 Series Pin Installation


...

,Material No.,Item and Spec,Units,Remark,Section
14,1010002078,Gear pump,1,2ABPF04LJ36SB09L-TLT,ZS0608/ZS0808/ZS0812 Gear Pump Installation
15,1140114035,Hydraulic pipe joint \n (ZS0608),1,M18×1.5-3/4-16UNF,ZS0608/ZS0808/ZS0812 Gear Pump Installation
16,1140103421,Male stud connector \n(ZS0808/ZS0812),1,GE12L7/8UNFOMDCF,ZS0608/ZS0808/ZS0812 Gear Pump Installation
17,1140114034,Hydraulic pipe joint,1,GE18L7/8UNFOMD,ZS0608/ZS0808/ZS0812 Gear Pump Installation
18,1040004607,Bolt,2,3/8-24UNF-1-10.9,ZS0608/ZS0808/ZS0812 Gear Pump Installation
19,1040300067,Gasket,2,GB/T93-1987,ZS0608/ZS0808/ZS0812 Gear Pump Installation


,Material No.,Item and Spec,Units,Remark,Section
14,1010002078,Gear pump,1,2ABPF05LJ36SB09L-TLT,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
15,1140114035,Hydraulic pipe joint,1,M18×1.5-3/4-16UNF,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
16,1140114034,Hydraulic pipe joint,1,GE18L7/8UNFOMD,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
17,1040004607,Bolt,2,3/8-24UNF-1-10.9,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
18,1040300067,Gasket,2,GB/T93-1987,ZS1012/ZS1212/ZS1414DC Gear Pump Installation


,Material No.,Item and Spec,Units,Remark,Section
0,00775700000001040,Lower sliding block I,1.0,NaN,ZS1414 Series Pin Installation
1,00775600300001140,Gasket,2.0,NaN,ZS1414 Series Pin Installation
2,1054400310,Composite bush,2.0,NaN,ZS1414 Series Pin Installation
3,1040200504,Nut M10-8,2.0,GB/T889.1-2000,ZS1414 Series Pin Installation
4,1040000025,Bolt M10×80-8.8,2.0,GB/T5783-2000,ZS1414 Series Pin Installation
...,...,...,...,...,...
2738,1010002078,Gear pump,1.0,2ABPF05LJ36SB09L-TLT,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
2739,1140114035,Hydraulic pipe joint,1.0,M18×1.5-3/4-16UNF,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
2740,1140114034,Hydraulic pipe joint,1.0,GE18L7/8UNFOMD,ZS1012/ZS1212/ZS1414DC Gear Pump Installation
2741,1040004607,Bolt,2.0,3/8-24UNF-1-10.9,ZS1012/ZS1212/ZS1414DC Gear Pump Installation


,Material No.,Item and Spec,Units,Remark,Section
0,00775700000001040,Lower sliding block I,1.0,NaN,ZS1414 Series Pin Installation
1,00775600300001140,Gasket,2.0,NaN,ZS1414 Series Pin Installation
2,1054400310,Composite bush,2.0,NaN,ZS1414 Series Pin Installation
3,1040200504,Nut M10-8,2.0,GB/T889.1-2000,ZS1414 Series Pin Installation
4,1040000025,Bolt M10×80-8.8,2.0,GB/T5783-2000,ZS1414 Series Pin Installation


100%|██████████| 1/1 [00:03<00:00,  3.76s/it]


- Спросить у Миши про запчасти, в описании которых встречается ещё одна модель, помимо основной, в которой они встречаются
- Что делать с такими в описании (Cubota): ZT138J-ZT24JE-V Turntable Decal Assembly - что значит тире?